# Bahrain Rental Price — Baseline Submission (Linear Regression)
Trains on `data.csv`, predicts on `test.csv`, and writes `submission.csv`.
This is the simple baseline that scored ~111 MAE on the leaderboard.

In [1]:
# Imports
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression

## 1. Small cleaning function
Pulls a number out of the three messy text columns. Used for both train and test.

In [2]:
def clean(df):
    df = df.copy()
    # Cast to string first so it works even if a column loads as numbers
    beds = df['Beds'].astype(str)
    baths = df['Baths'].astype(str)
    size = df['Size'].astype(str)
    # Beds: "studio" -> 0, "3+ Maid" -> 3, "2" -> 2
    df['Beds_num'] = beds.str.extract(r'(\d+)').astype(float)
    df.loc[beds.str.contains('studio', case=False, na=False), 'Beds_num'] = 0
    # Baths: "7+" -> 7
    df['Baths_num'] = baths.str.extract(r'(\d+)').astype(float)
    # Size: "2,368 sqft / 220 sqm" -> 220
    df['Size_sqm'] = size.str.extract(r'/\s*([\d,]+)\s*sqm')[0].str.replace(',', '').astype(float)
    return df

## 2. Load and clean the training data

In [3]:
# Load training data
df = pd.read_csv('data.csv')

# Drop the few rows with no rent (our target)
df = df.dropna(subset=['rent'])

# Clean it
df = clean(df)

# Features
num_cols = ['Beds_num', 'Baths_num', 'Size_sqm', 'Amenities']
cat_cols = ['Property_type', 'Governorate', 'Include_w_e']

X = df[num_cols + cat_cols]
y = df['rent']

## 3. Build the pipeline
Numbers → impute median → scale. Text → impute most frequent → one-hot. Then Linear Regression.

In [4]:
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('lr', LinearRegression())
])

## 4. Train
We ignore the top 1% highest rents while fitting, since a few extreme values (up to 400,000) would wreck a linear model.

In [5]:
# Fit on all data, ignoring the top 1% extreme rents
keep = y < y.quantile(0.99)
model.fit(X[keep], y[keep])
print('Model trained.')

Model trained.


## 5. Predict on the test set and save the submission

In [6]:
# Load the test file
test = pd.read_csv('test.csv')

# Clean it the same way
test_clean = clean(test)

# Predict
pred = model.predict(test_clean[num_cols + cat_cols])

# Save in the required format: Property_id, rent
submission = pd.DataFrame({
    'Property_id': test['Property_id'],
    'rent': pred
})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv with', len(submission), 'rows')
submission.head()

Saved submission.csv with 3527 rows


,Property_id,rent
0,4394,471.297685
1,2338,499.889969
2,8531,977.124709
3,8952,360.361098
4,11064,190.878229
